In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import swin_t

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve,
    precision_recall_curve
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
data_dir = "/kaggle/input/datasets/afridirahman/brain-stroke-ct-image-dataset/Brain_Data_Organised"

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [ ]:
dataset = datasets.ImageFolder(data_dir, transform=transform)

labels = np.array([dataset.samples[i][1] for i in range(len(dataset))])

print("Classes:", dataset.classes)
print("Class distribution:", np.bincount(labels))

In [ ]:
train_idx, temp_idx = train_test_split(
    np.arange(len(labels)),
    test_size=0.3,
    stratify=labels,
    random_state=42
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=labels[temp_idx],
    random_state=42
)

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)
test_dataset = Subset(dataset, test_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=3e-5)

In [ ]:
class SwinTinyModel(nn.Module):
    def __init__(self):
        super(SwinTinyModel, self).__init__()

        self.backbone = swin_t(weights="IMAGENET1K_V1")

        num_features = self.backbone.head.in_features

        self.backbone.head = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_features, 1)
        )

    def forward(self, x):
        return self.backbone(x)

model = SwinTinyModel().to(device)

In [ ]:
num_normal = np.sum(labels == 0)
num_stroke = np.sum(labels == 1)

pos_weight = torch.tensor([num_normal / num_stroke]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=3e-5)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5
)

In [ ]:
epochs = 20

for epoch in range(epochs):

    model.train()
    train_loss = 0

    for images, labels_batch in train_loader:

        images = images.to(device)
        labels_batch = labels_batch.float().unsqueeze(1).to(device)

        optimizer.zero_grad()

        outputs = model(images)
        outputs = torch.clamp(outputs, -50, 50)

        loss = criterion(outputs, labels_batch)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for images, labels_batch in val_loader:
            images = images.to(device)
            labels_batch = labels_batch.float().unsqueeze(1).to(device)

            outputs = model(images)
            outputs = torch.clamp(outputs, -50, 50)

            loss = criterion(outputs, labels_batch)
            val_loss += loss.item()

    scheduler.step(val_loss)

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {train_loss/len(train_loader):.4f} | "
          f"Val Loss: {val_loss/len(val_loader):.4f}")

In [ ]:
model.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels_batch in test_loader:

        images = images.to(device)

        outputs = model(images)
        outputs = torch.clamp(outputs, -50, 50)

        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()

        all_preds.extend(preds.cpu().numpy().ravel())
        all_labels.extend(labels_batch.numpy())
        all_probs.extend(probs.cpu().numpy().ravel())

In [ ]:
acc = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds)
rec = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
auc = roc_auc_score(all_labels, all_probs)

print("\nCLASSIFICATION REPORT:\n")
print(classification_report(all_labels, all_preds))

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1 Score:", f1)
print("AUC:", auc)

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=["Normal","Stroke"],
            yticklabels=["Normal","Stroke"],
            cmap="Blues")

plt.title("Confusion Matrix - Swin Tiny")
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(all_labels, all_probs)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
plt.plot([0,1],[0,1],'--')
plt.legend()
plt.title("ROC Curve - Swin Tiny")
plt.show()

In [ ]:
model.eval()
count = 0
plt.figure(figsize=(12,8))

with torch.no_grad():
    for images, labels_batch in test_loader:

        images = images.to(device)
        outputs = model(images)
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()

        for i in range(images.size(0)):

            img = images[i].cpu().permute(1,2,0).numpy()
            img = img * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406])
            img = np.clip(img, 0, 1)

            true_label = "Stroke" if labels_batch[i]==1 else "Normal"
            pred_label = "Stroke" if preds[i]==1 else "Normal"

            plt.subplot(2,3,count+1)
            plt.imshow(img)
            plt.title(f"True: {true_label}\nPred: {pred_label} ({probs[i].item():.2f})")
            plt.axis("off")

            count += 1
            if count == 6:
                break
        if count == 6:
            break

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import StratifiedKFold

In [ ]:
class SwinTinyModel(nn.Module):
    def __init__(self):
        super(SwinTinyModel, self).__init__()

        self.backbone = swin_t(weights="IMAGENET1K_V1")

        num_features = self.backbone.head.in_features

        self.backbone.head = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_features, 1)
        )

    def forward(self, x):
        return self.backbone(x)

In [ ]:
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
fold_results = {
    "accuracy": [],
    "recall": [],
    "f1": [],
    "auc": []
}

for fold, (train_idx, val_idx) in enumerate(
        kfold.split(np.arange(len(dataset)), labels)):

    print(f"\n===== FOLD {fold+1} =====")

    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)

    model = SwinTinyModel().to(device)

    # Weighted loss per fold
    num_normal = np.sum(labels[train_idx] == 0)
    num_stroke = np.sum(labels[train_idx] == 1)

    pos_weight = torch.tensor([num_normal / num_stroke]).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=3e-5)

    epochs = 10   # Increase to 15–20 for stronger training

    # ---- Training ----
    for epoch in range(epochs):
        model.train()

        for images, labels_batch in train_loader:

            images = images.to(device)
            labels_batch = labels_batch.float().unsqueeze(1).to(device)

            optimizer.zero_grad()

            outputs = model(images)
            outputs = torch.clamp(outputs, -50, 50)

            loss = criterion(outputs, labels_batch)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

    # ---- Validation ----
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for images, labels_batch in val_loader:

            images = images.to(device)

            outputs = model(images)
            outputs = torch.clamp(outputs, -50, 50)

            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()

            all_preds.extend(preds.cpu().numpy().ravel())
            all_labels.extend(labels_batch.numpy())
            all_probs.extend(probs.cpu().numpy().ravel())

    acc = accuracy_score(all_labels, all_preds)
    rec = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)

    fold_results["accuracy"].append(acc)
    fold_results["recall"].append(rec)
    fold_results["f1"].append(f1)
    fold_results["auc"].append(auc)

    print(f"Accuracy: {acc:.4f}")
    print(f"Recall: {rec:.4f}")
    print(f"F1: {f1:.4f}")
    print(f"AUC: {auc:.4f}")

In [ ]:
print("\n===== FINAL 5-FOLD RESULTS =====")

for metric in fold_results:
    values = fold_results[metric]
    print(f"{metric.upper()}: {np.mean(values):.4f} ± {np.std(values):.4f}")

In [ ]:
import cv2

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0]

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def generate(self, input_image):
        self.model.eval()

        output = self.model(input_image)
        self.model.zero_grad()
        output.backward(torch.ones_like(output))

        gradients = self.gradients[0]
        activations = self.activations[0]

        # Swin outputs [B, H, W, C] sometimes → fix dimension
        if gradients.dim() == 4:
            gradients = gradients.permute(2, 0, 1)
            activations = activations.permute(2, 0, 1)

        weights = torch.mean(gradients, dim=(1,2))

        cam = torch.zeros(activations.shape[1:], dtype=torch.float32).to(device)

        for i, w in enumerate(weights):
            cam += w * activations[i]

        cam = torch.relu(cam)
        cam = cam.detach().cpu().numpy()

        cam -= cam.min()
        cam /= (cam.max() + 1e-8)

        return cam

In [ ]:
target_layer = model.backbone.features[-1][-1].norm2
gradcam = GradCAM(model, target_layer)

In [ ]:
model.eval()

plt.figure(figsize=(8,6))
count = 0

for images, labels_batch in test_loader:

    for i in range(len(labels_batch)):

        if labels_batch[i].item() == 1:  # Stroke class

            input_image = images[i].unsqueeze(0).to(device)
            original_image = images[i]

            heatmap = gradcam.generate(input_image)

            img = original_image.cpu().permute(1,2,0).numpy()
            img = img * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406])
            img = np.clip(img, 0, 1)

            heatmap = cv2.resize(heatmap, (224,224))
            heatmap = np.uint8(255 * heatmap)
            heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
            heatmap = heatmap.astype(np.float32) / 255

            overlay = heatmap * 0.4 + img

            plt.subplot(1,2,count+1)
            plt.imshow(overlay)
            plt.axis("off")

            count += 1

            if count == 2:
                break
    if count == 2:
        break

plt.show()